<a href="https://colab.research.google.com/github/essanchristian-maker/DI-Bootcamp/blob/master/Week8_Day4_Exercises_XP_Gold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
# Ensure the MCP library is installed in the Colab environment
!pip install "mcp[cli]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 14.3 MB/s eta 0:00:00


### Part A & B: Server Implementation
This cell defines the `server_stream.py` logic. Note that `ctx.info()` is used to send progress notifications.

In [5]:
content = """
from mcp.server.fastmcp import FastMCP
import asyncio

# Initialize FastMCP server
mcp = FastMCP("Streaming-Demo")

@mcp.tool(description="Simulates a long-running task with progress updates.")
async def process_items(total: int = 5, ctx=None) -> str:
    for i in range(1, total + 1):
        await asyncio.sleep(0.3)  # Brief sleep for demo purposes

        # Emit structured progress via the context object if available
        if ctx:
            await ctx.info(f"Processing item {i} of {total}...")
        else:
            print(f"[Fallback Log] Processing {i}/{total}")

    return f"Successfully processed {total} items."

if __name__ == '__main__':
    mcp.run()
"""

with open('server_stream.py', 'w') as f:
    f.write(content)
print("server_stream.py created successfully.")

server_stream.py created successfully.


### Part C: Client Implementation
This client connects to the server, handles incoming notifications, and displays the final result.

In [8]:
content = """
import asyncio
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_client():
    # Configure the server to run via STDIO
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["server_stream.py"],
        env=None
    )

    print("--- Starting MCP Client ---")

    async with stdio_client(server_params) as (read, write):
        async def handle_notification(notification):
            # Capture logs from FastMCP ctx.info()
            if hasattr(notification, 'params') and hasattr(notification.params, 'message'):
                print(f"NOTIFICATION: {notification.params.message}")
            elif hasattr(notification, 'method') and "log" in notification.method:
                print(f"NOTIFICATION (Log): {notification.params}")

        async with ClientSession(read, write) as session:
            await session.initialize()
            session.on_notification = handle_notification

            print("Calling tool: process_items(total=5)...\\n")

            result = await session.call_tool("process_items", arguments={"total": 5})

            print("\\n--- Final Result ---")
            print(result.content[0].text)

if __name__ == "__main__":
    asyncio.run(run_client())
"""

with open('client_stream.py', 'w') as f:
    f.write(content)
print("client_stream.py updated and fixed.")

client_stream.py updated and fixed.


### Execution
Run the cell below to execute the client and see the streaming progress in action.

In [9]:
!python client_stream.py

--- Starting MCP Client ---
Calling tool: process_items(total=5)...

[07/17/26 01:43:55] INFO     Processing request of type            ]8;id=628998;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=782084;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             CallToolRequest                                    
[07/17/26 01:43:57] INFO     Processing request of type            ]8;id=67069;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=350989;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   

--- Final Result ---
Successfully processed 5 items.
